# GĐ3 — Đánh giá trên Kaggle T4 ×2: ablation, thay N, mạng chậm (VGG-16 / CIFAR-100)
Yêu cầu: Accelerator = **GPU T4 x2**, Internet = **On**, dataset CIFAR-100 như GĐ2.
Thiết kế: paper_notes §23. Baseline (TiMePReSt graph, PipeDream, lr 0.02) đã có ở `results/phase2/run2_lr0.02_timeprest-graph`.
Chạy lần lượt; dán cho agent output cell 3 (check), dòng cuối mỗi run, bảng của cell E3, rồi tải `results_phase3.zip`.
Nếu session bị ngắt: chạy lại cell 1, rồi chạy lại đúng cell đang dở (các lệnh train có `--resume`).

## 1. Repo + cài đặt + môi trường

In [ ]:
import os
REPO = '/kaggle/working/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count(), '| nccl', torch.cuda.nccl.version())"
!find /kaggle/input -maxdepth 4 -name cifar-100-python

## 2. Unit test (CPU) ~2–3 phút

In [ ]:
!python -m pytest -q

## 3. Check GĐ3 (D1–D4 cho pipedream, timeprest, variant1, variant2) ~2 phút
Phải ra `ALL PHASE-2 CHECKS PASS: True` thì các lệnh chạy dài mới chạy.

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.checks --config configs/kaggle_quick_gd3.yaml

## 4. E1 — Ablation paper §4.10 (~0.75 h mỗi run)
Variant 1: nF1B + **giữ** weight stashing. Variant 2: **1F1B**, bỏ weight stashing.

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/kaggle_variant1.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/kaggle_variant2.yaml --resume

## 5. E2 — Thay N, paper §4.9 (~0.7 h mỗi run)
N=2 cùng M=192; N=2 với M=384.

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/kaggle_timeprest_n2.yaml --resume

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.train --config configs/kaggle_timeprest_n2_m384.yaml --resume

## 6. E3 — Thời gian/epoch theo băng thông giả lập (~30–45 phút)
Mỗi mức băng thông × {TiMePReSt, PipeDream}: 2 epoch trên toàn bộ tập train, đo epoch cuối. Dán bảng cuối cho agent.

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.dist.bench_comm --config configs/kaggle_bench_comm.yaml
from IPython.display import Image
display(Image('/kaggle/working/bench_comm/bench_comm.png'))

## 7. Đóng gói kết quả để tải về

In [ ]:
RUNS = '/kaggle/working/runs'
!cd /kaggle/working && zip -r /kaggle/working/results_phase3.zip runs/*/metrics.csv runs/*/summary.json runs/*/config.json runs/*/op_trace_epoch1.json runs/*/checks.json bench_comm/bench_comm.* bench_comm/*/metrics.csv bench_comm/*/config.json
from IPython.display import FileLink
display(FileLink('/kaggle/working/results_phase3.zip'))